<a href="https://colab.research.google.com/github/Reyy15-09/KKA/blob/main/kerkomkka.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# ============================================================
# PROYEK AKHIR: EDA STARTER PROJECT
# Dataset: Nilai Akademik Siswa
# ============================================================

# Import library
import pandas as pd
import numpy as np
from datetime import datetime

# Agar output lebih rapi
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 40)

# --- Data Loading ---
# Upload file CSV ke Colab, lalu jalankan:
# from google.colab import files
# uploaded = files.upload()

# Atau jika sudah di Drive / path lokal:
df = pd.read_csv('dataset_nilai_akademik_siswa (1).csv')   # ganti nama file jika berbeda

print("Dataset berhasil dimuat!")
print(f"Shape: {df.shape}")

Dataset berhasil dimuat!
Shape: (79, 8)


In [5]:
# ============================================================
# DATA INSPECTION
# ============================================================

print("=" * 60)
print("1. HEAD (5 baris pertama)")
print("=" * 60)
display(df.head())

print("\n" + "=" * 60)
print("2. INFO (tipe data & non-null count)")
print("=" * 60)
df.info()

print("\n" + "=" * 60)
print("3. DESCRIBE (statistik deskriptif)")
print("=" * 60)
display(df.describe(include='all'))

print("\n" + "=" * 60)
print("4. SHAPE")
print("=" * 60)
print(f"Jumlah baris : {df.shape[0]}")
print(f"Jumlah kolom : {df.shape[1]}")

print("\n" + "=" * 60)
print("5. CEK MISSING VALUE")
print("=" * 60)
print(df.isnull().sum())

print("\n" + "=" * 60)
print("6. CEK DUPLIKAT")
print("=" * 60)
print(f"Jumlah baris duplikat: {df.duplicated().sum()}")

1. HEAD (5 baris pertama)


,id_siswa,nama,kelas,mata_pelajaran,jenis_ujian,tanggal_ujian,nilai,guru_pengampu
0,SIS0004,Joko Prasetyo,XI RPL 2,KKA,uas,06/08/2026,46,Ibu Wati
1,SIS0020,Eka Putri,XI RPL 1,PKK,uts,12 Agustus 2026,"73,0",Bpk. Santoso
2,SIS0015,Nanda Pratama,XI RPL 3,Pemrograman Web,UAS,2026-08-15,69,Bpk. Santoso
3,SIS0046,Fajar Nugroho,XI RPL 2,Matematika,UH,10/08/2026,73,Bpk. Arifin
4,SIS0011,Ayu Lestari,XI RPL 2,Pemrograman Web,UH,6 Agustus 2026,53,Ibu Wati



2. INFO (tipe data & non-null count)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 79 entries, 0 to 78
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id_siswa        79 non-null     object
 1   nama            79 non-null     object
 2   kelas           79 non-null     object
 3   mata_pelajaran  79 non-null     object
 4   jenis_ujian     79 non-null     object
 5   tanggal_ujian   79 non-null     object
 6   nilai           75 non-null     object
 7   guru_pengampu   75 non-null     object
dtypes: object(8)
memory usage: 5.1+ KB

3. DESCRIBE (statistik deskriptif)


,id_siswa,nama,kelas,mata_pelajaran,jenis_ujian,tanggal_ujian,nilai,guru_pengampu
count,79,79,79,79,79,79,75,75
unique,75,20,3,6,6,38,52,5
top,SIS0046,Lukman Hakim,XI RPL 3,PKK,UH,2026-08-05,73,Bpk. Arifin
freq,2,7,34,16,21,8,4,18



4. SHAPE
Jumlah baris : 79
Jumlah kolom : 8

5. CEK MISSING VALUE
id_siswa          0
nama              0
kelas             0
mata_pelajaran    0
jenis_ujian       0
tanggal_ujian     0
nilai             4
guru_pengampu     4
dtype: int64

6. CEK DUPLIKAT
Jumlah baris duplikat: 4


In [6]:
# ============================================================
# DATA CLEANING
# ============================================================

# Buat salinan agar data asli tetap aman
df_clean = df.copy()

# ----------------------------------------------------------
# 3.1 Hapus Duplikat
# ----------------------------------------------------------
print("Sebelum hapus duplikat:", df_clean.shape)
df_clean = df_clean.drop_duplicates()
print("Setelah hapus duplikat :", df_clean.shape)
print("Alasan: Baris yang benar-benar identik dihapus agar tidak bias analisis.\n")

# ----------------------------------------------------------
# 3.2 Bersihkan kolom NILAI
# ----------------------------------------------------------
# Hapus kata "poin", ganti koma menjadi titik, ubah ke numeric
df_clean['nilai'] = (
    df_clean['nilai']
    .astype(str)
    .str.replace('poin', '', case=False, regex=False)
    .str.replace(',', '.', regex=False)
    .str.strip()
)

# Ubah ke numeric (yang tidak bisa diubah jadi NaN)
df_clean['nilai'] = pd.to_numeric(df_clean['nilai'], errors='coerce')

# Tangani outlier 999 (tidak masuk akal untuk nilai 0-100)
print("Nilai outlier (nilai > 100):")
print(df_clean[df_clean['nilai'] > 100][['id_siswa', 'nama', 'nilai']])

# Ganti 999 menjadi NaN (akan ditangani bersama missing value)
df_clean.loc[df_clean['nilai'] > 100, 'nilai'] = np.nan

print("\nAlasan: 999 merupakan error input. Lebih aman diubah ke NaN lalu diputuskan apakah diisi atau dihapus.")

# ----------------------------------------------------------
# 3.3 Tangani Missing Value
# ----------------------------------------------------------
print("\nMissing value sebelum ditangani:")
print(df_clean.isnull().sum())

# Strategi:
# - nilai kosong: diisi dengan median per mata_pelajaran (lebih representatif daripada mean)
# - guru_pengampu kosong: diisi "Tidak diketahui" (karena informasi kategori)

median_per_mapel = df_clean.groupby('mata_pelajaran')['nilai'].transform('median')
df_clean['nilai'] = df_clean['nilai'].fillna(median_per_mapel)

# Jika masih ada NaN (mapel yang semua nilainya kosong), isi dengan median global
df_clean['nilai'] = df_clean['nilai'].fillna(df_clean['nilai'].median())

df_clean['guru_pengampu'] = df_clean['guru_pengampu'].fillna('Tidak diketahui')

print("\nMissing value setelah ditangani:")
print(df_clean.isnull().sum())
print("\nAlasan pilihan teknik:")
print("- fillna median per mapel: mempertahankan distribusi nilai tiap mata pelajaran")
print("- fillna 'Tidak diketahui' untuk guru: informasi kategori, tidak bisa diisi angka")

# ----------------------------------------------------------
# 3.4 Normalisasi jenis_ujian
# ----------------------------------------------------------
df_clean['jenis_ujian'] = df_clean['jenis_ujian'].str.upper().str.strip()
print("\nJenis ujian setelah dinormalisasi:")
print(df_clean['jenis_ujian'].unique())

# ----------------------------------------------------------
# 3.5 Parsing tanggal_ujian ke datetime
# ----------------------------------------------------------
def parse_tanggal(tgl):
    tgl = str(tgl).strip()
    # Format yang mungkin muncul
    formats = [
        '%d/%m/%Y',
        '%Y-%m-%d',
        '%d Agustus %Y',
        '%d Agustus%Y',
        '%d Agustus  %Y'
    ]
    for fmt in formats:
        try:
            return pd.to_datetime(tgl, format=fmt)
        except:
            continue
    # Fallback: coba parse otomatis
    try:
        return pd.to_datetime(tgl, dayfirst=True)
    except:
        return pd.NaT

df_clean['tanggal_ujian'] = df_clean['tanggal_ujian'].apply(parse_tanggal)

print("\nContoh tanggal setelah parsing:")
print(df_clean[['tanggal_ujian']].head(8))

# ----------------------------------------------------------
# 3.6 Pastikan tipe data final
# ----------------------------------------------------------
df_clean['nilai'] = df_clean['nilai'].astype(float)
df_clean['id_siswa'] = df_clean['id_siswa'].astype(str)

print("\n" + "=" * 60)
print("INFO SETELAH CLEANING")
print("=" * 60)
df_clean.info()

print("\nShape akhir setelah cleaning:", df_clean.shape)

Sebelum hapus duplikat: (79, 8)
Setelah hapus duplikat : (75, 8)
Alasan: Baris yang benar-benar identik dihapus agar tidak bias analisis.

Nilai outlier (nilai > 100):
   id_siswa           nama  nilai
10  SIS0023  Hendra Wijaya  999.0

Alasan: 999 merupakan error input. Lebih aman diubah ke NaN lalu diputuskan apakah diisi atau dihapus.

Missing value sebelum ditangani:
id_siswa          0
nama              0
kelas             0
mata_pelajaran    0
jenis_ujian       0
tanggal_ujian     0
nilai             5
guru_pengampu     3
dtype: int64

Missing value setelah ditangani:
id_siswa          0
nama              0
kelas             0
mata_pelajaran    0
jenis_ujian       0
tanggal_ujian     0
nilai             0
guru_pengampu     0
dtype: int64

Alasan pilihan teknik:
- fillna median per mapel: mempertahankan distribusi nilai tiap mata pelajaran
- fillna 'Tidak diketahui' untuk guru: informasi kategori, tidak bisa diisi angka

Jenis ujian setelah dinormalisasi:
['UAS' 'UTS' 'UH']

Conto

In [7]:
# ============================================================
# DATA MANIPULATION
# ============================================================

# ----------------------------------------------------------
# 4.1 FILTERING
# Contoh: Siswa yang mendapat nilai di bawah 60 (perlu perhatian)
# ----------------------------------------------------------
df_nilai_rendah = df_clean[df_clean['nilai'] < 60].copy()
print("Jumlah data nilai < 60:", len(df_nilai_rendah))
display(df_nilai_rendah[['nama', 'kelas', 'mata_pelajaran', 'jenis_ujian', 'nilai']].head(10))

# ----------------------------------------------------------
# 4.2 SORTING
# Urutkan berdasarkan nilai tertinggi
# ----------------------------------------------------------
df_sorted = df_clean.sort_values(by='nilai', ascending=False)
print("\nTop 10 nilai tertinggi:")
display(df_sorted[['nama', 'kelas', 'mata_pelajaran', 'jenis_ujian', 'nilai']].head(10))

# ----------------------------------------------------------
# 4.3 KOLOM TURUNAN (Derived Column)
# Buat kategori prestasi berdasarkan nilai
# ----------------------------------------------------------
def kategori_prestasi(nilai):
    if nilai >= 85:
        return 'Sangat Baik'
    elif nilai >= 70:
        return 'Baik'
    elif nilai >= 60:
        return 'Cukup'
    else:
        return 'Perlu Perbaikan'

df_clean['kategori_prestasi'] = df_clean['nilai'].apply(kategori_prestasi)

print("\nDistribusi kategori prestasi:")
print(df_clean['kategori_prestasi'].value_counts())

# ----------------------------------------------------------
# 4.4 GROUPBY / AGREGASI
# Rata-rata nilai per kelas dan mata pelajaran
# ----------------------------------------------------------
agg_kelas_mapel = (
    df_clean
    .groupby(['kelas', 'mata_pelajaran'])
    .agg(
        rata_rata_nilai=('nilai', 'mean'),
        jumlah_data=('nilai', 'count'),
        nilai_tertinggi=('nilai', 'max'),
        nilai_terendah=('nilai', 'min')
    )
    .round(2)
    .reset_index()
)

print("\nAgregasi rata-rata nilai per Kelas + Mata Pelajaran:")
display(agg_kelas_mapel)

# Agregasi tambahan: rata-rata per jenis ujian
agg_jenis = (
    df_clean
    .groupby('jenis_ujian')['nilai']
    .agg(['mean', 'median', 'count'])
    .round(2)
)
print("\nRata-rata nilai berdasarkan jenis ujian:")
display(agg_jenis)

Jumlah data nilai < 60: 30


,nama,kelas,mata_pelajaran,jenis_ujian,nilai
0,Joko Prasetyo,XI RPL 2,KKA,UAS,46.0
4,Ayu Lestari,XI RPL 2,Pemrograman Web,UH,53.0
5,Maya Anggraini,XI RPL 2,PKK,UTS,42.0
6,Gita Ramadhani,XI RPL 3,Bahasa Inggris,UTS,45.0
9,Eka Putri,XI RPL 1,PKK,UH,56.0
11,Citra Ningrum,XI RPL 3,Bahasa Inggris,UAS,56.0
15,Joko Prasetyo,XI RPL 3,Matematika,UH,56.0
18,Rina Marlina,XI RPL 3,Pemrograman Web,UTS,44.0
19,Gita Ramadhani,XI RPL 1,Matematika,UTS,45.0
21,Satria Nugraha,XI RPL 1,KKA,UAS,54.0



Top 10 nilai tertinggi:


,nama,kelas,mata_pelajaran,jenis_ujian,nilai
34,Hendra Wijaya,XI RPL 2,Matematika,UH,100.0
8,Putri Amelia,XI RPL 1,Pemrograman Web,UH,98.0
61,Eka Putri,XI RPL 2,KKA,UTS,96.0
40,Ayu Lestari,XI RPL 3,PKK,UH,92.0
22,Lukman Hakim,XI RPL 1,Pemrograman Web,UH,91.0
28,Dedi Saputra,XI RPL 2,PKK,UH,90.0
63,Kirana Salsabila,XI RPL 2,Bahasa Inggris,UTS,90.0
43,Lukman Hakim,XI RPL 2,Pemrograman Web,UH,88.0
26,Indah Permata,XI RPL 2,KKA,UH,87.0
66,Lukman Hakim,XI RPL 1,PKK,UAS,84.0



Distribusi kategori prestasi:
kategori_prestasi
Perlu Perbaikan    30
Cukup              21
Baik               15
Sangat Baik         9
Name: count, dtype: int64

Agregasi rata-rata nilai per Kelas + Mata Pelajaran:


,kelas,mata_pelajaran,rata_rata_nilai,jumlah_data,nilai_tertinggi,nilai_terendah
0,XI RPL 1,Bahasa Inggris,70.25,2,71.0,69.5
1,XI RPL 1,Basis Data,66.33,3,74.0,62.0
2,XI RPL 1,KKA,56.50,2,59.0,54.0
3,XI RPL 1,Matematika,60.50,2,76.0,45.0
4,XI RPL 1,PKK,66.40,5,84.0,44.0
5,XI RPL 1,Pemrograman Web,72.67,6,98.0,42.0
6,XI RPL 2,Bahasa Inggris,81.50,2,90.0,73.0
7,XI RPL 2,Basis Data,53.80,5,70.0,43.0
8,XI RPL 2,KKA,74.75,4,96.0,46.0
9,XI RPL 2,Matematika,70.00,5,100.0,44.0



Rata-rata nilai berdasarkan jenis ujian:


,mean,median,count
jenis_ujian,,,
UAS,61.08,60.5,26
UH,70.19,67.0,27
UTS,61.43,65.0,22


In [8]:
# ============================================================
# SIMPAN DATASET BERSIH
# ============================================================

df_clean.to_csv('dataset_bersih.csv', index=False)
print("Dataset bersih berhasil disimpan sebagai 'dataset_bersih.csv'")
print("File ini siap digunakan untuk Elemen 3 (Visualisasi Data)")

Dataset bersih berhasil disimpan sebagai 'dataset_bersih.csv'
File ini siap digunakan untuk Elemen 3 (Visualisasi Data)


In [9]:
# ============================================================
# DATA PROFILING SUMMARY
# ============================================================

print("""
RINGKASAN TEMUAN AWAL (siap untuk presentasi)

1. Setelah pembersihan data, terdapat sekitar 75 baris data valid.
   Mayoritas siswa berada di kelas XI RPL 3. Mata pelajaran yang paling
   sering muncul adalah PKK dan Matematika.

2. Terdapat cukup banyak siswa dengan nilai di bawah 60 (kategori
   "Perlu Perbaikan"). Ini mengindikasikan adanya tantangan pembelajaran
   pada beberapa mata pelajaran, khususnya yang berkaitan dengan praktik
   (seperti Basis Data dan Pemrograman Web).

3. Rata-rata nilai cenderung lebih tinggi pada jenis ujian UH dibandingkan
   UAS/UTS. Hal ini wajar karena UH biasanya mencakup materi yang lebih
   sempit. Namun, perlu perhatian lebih pada siswa yang konsisten mendapat
   nilai rendah lintas jenis ujian.
""")


RINGKASAN TEMUAN AWAL (siap untuk presentasi)

1. Setelah pembersihan data, terdapat sekitar 75 baris data valid.
   Mayoritas siswa berada di kelas XI RPL 3. Mata pelajaran yang paling
   sering muncul adalah PKK dan Matematika.

2. Terdapat cukup banyak siswa dengan nilai di bawah 60 (kategori
   "Perlu Perbaikan"). Ini mengindikasikan adanya tantangan pembelajaran
   pada beberapa mata pelajaran, khususnya yang berkaitan dengan praktik
   (seperti Basis Data dan Pemrograman Web).

3. Rata-rata nilai cenderung lebih tinggi pada jenis ujian UH dibandingkan
   UAS/UTS. Hal ini wajar karena UH biasanya mencakup materi yang lebih
   sempit. Namun, perlu perhatian lebih pada siswa yang konsisten mendapat
   nilai rendah lintas jenis ujian.

